# 11 - Pass 5: down-weighted and cleaned generated pairs (one commit, GPU)

This notebook trains every Pass 5 run in **one Save & Run All (Commit)**. Same structure as notebook 08: it fetches the code from GitHub, runs the tests, checks the data on the CPU, then trains two runs at a time (one per GPU).

**What Pass 5 asks** (vault: Experiment Plan, "Pass 5 plan"): Pass 4's pairs cost 15-16 points on Headlines. Does trusting the pairs less (`pair_weight` = 0.1) and/or dropping the pairs that lean most on AI writing habits (worst 20% of seeds) win that back, and what happens to the iSarcasmEval transfer gain?

Both settings were picked on dev in notebook 10: `PASS5_PAIR_WEIGHT = 0.1`, `PASS5_CLEANED_DROP_PCT = 20`.

| Arm | What it trains on |
|---|---|
| A16 | labeled headlines + cleaned pairs (worst 20% dropped), pair weight 0.1 |
| A17 | A16 + FGM (the adversarial push from A7) |
| A10, tag `weighted` | labeled headlines + all pairs (no cleaning), pair weight 0.1: the weight alone |

5 seeds each, at 10% and 1% labels = 30 runs, test scored, masked test sets scored. Baselines A1, A7 (Passes 1-2) and A10, A11 (Pass 4) are reused, not re-run.

**Before you run, in the right-hand panel:**
- **Input:** the `gainsplit-processed` version with `pairs_train.csv`, `manifest_pairs.json`, `pair_shortcut_scores.csv` and `manifest_shortcut_scores.json` (the same version notebook 10 used).
- **Internet: on** (code from GitHub, BERT from Hugging Face).
- **Accelerator: GPU T4 x2.**
- Then **Save Version -> Save & Run All (Commit)**, so the output is kept after the session ends.


## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about a minute)
The repo's tests train a tiny random BERT for a few steps on made-up data, including the pair arms, the pair batches and the explanation step. They prove the pieces fit together before real GPU time is spent. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the data on the CPU (pre-flight)
1. Finds `gainsplit-processed` and checks every file against `manifest.json` and `manifest_pass3.json` (masked test sets).
2. Loads each pair set the runs use and checks it (hash against its manifest, one sarcastic and one serious row per seed, seeds from the training split only, no generated headline equal to a dev or test headline).
3. Prints the size of each set. Expected: `unfiltered` 44,834 rows (22,417 pairs); `cleaned` (20% dropped) about 35,868 rows (17,934 pairs).

Any problem raises an error here, before training starts.


In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass5"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN]()

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
from gainsplit.plans import PASS5_PAIR_WEIGHT, PASS5_CLEANED_DROP_PCT
print("frozen settings: pair_weight =", PASS5_PAIR_WEIGHT, "| cleaned_drop_pct =", PASS5_CLEANED_DROP_PCT)
display(preflight(configs, DATA_DIR))

## 4. The list of runs and the time estimate
`pass5` = 30 runs, seeds 0-4: A16, A17 and A10 (`weighted`) at 10% and at 1%.

Estimated run time (measured in notebooks 08 and 10): 10% about 2.3 min per run (A17 about 3.3), 1% about 0.7 min (A17 about 1). Total about 50 run-minutes, so **about 25-30 minutes on two GPUs**, plus about 5 minutes for steps 1-3.


In [ ]:
for c in configs:
    print(f"{c.run_id:24s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} eps={c.fgm_epsilon:g} "
          f"pair_weight={c.pair_weight:g} cleaned_drop_pct={c.cleaned_drop_pct:g}")
print(len(configs), "runs")


## 5. Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`. If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## 6. First look (the full analysis runs later on the CPU)
Mean test macro-F1, cue-masked F1 and iSarcasmEval AUC (points) per arm and budget, next to the baselines read from the repo clone: A1, A7 (Passes 1-2) and the unweighted A10, A11 (Pass 4).

What to look for: are A16 / A10-weighted much closer to A1 than Pass 4's A10 (about 64 at 10%)? Does iSarcasmEval AUC stay above about 69 (the length-only baseline) or fall towards 50?


In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for f in sorted(Path(OUT_DIR).glob("runs/*/result.json")):
    r = json.loads(f.read_text()); c = r["config"]
    rows.append({"arm": c["arm"], "budget": c["budget"], "seed": c["seed"], "tag": c.get("tag", ""),
                 "test_f1": r["scores"]["test"]["macro_f1"] * 100,
                 "cuemask_f1": r["scores"].get("test_cuemask", {}).get("macro_f1", float("nan")) * 100,
                 "isarc_auc": r["scores"]["isarc_test"]["roc_auc"] * 100,
                 "cue_share": (r.get("explain") or {}).get("mean_cue_share"), "minutes": r["minutes"]})
res = pd.DataFrame(rows)
display(res.groupby(["budget", "arm", "tag"])[["test_f1", "cuemask_f1", "isarc_auc", "cue_share", "minutes"]]
           .mean().round(3))

base = []
for p in ("pass1", "pass2b", "pass4"):
    s = Path(REPO_DIR) / "results" / p / "summary.csv"
    if s.exists():
        base.append(pd.read_csv(s).assign(source=p))
if base:
    base = pd.concat(base, ignore_index=True)
    base = base[base["arm"].isin(["A1", "A7", "A10", "A11"]) & base["budget"].isin([1, 10])]
    base = base[base["tag"].fillna("").eq("")]   # plain runs only (no 3x / explain / mask re-runs)
    cols = {"test_macro_f1": "test_f1", "isarc_test_roc_auc": "isarc_auc"}
    show = base.rename(columns=cols)
    keep = [c for c in ["test_f1", "isarc_auc"] if c in show]
    display((show.groupby(["budget", "arm", "source"])[keep].mean() * 100).round(3)
            .rename(columns=lambda c: c + " (baseline)"))


## 7. Download
Zips the results folder. A commit keeps it in the Output tab: download `results_<plan>.zip` and send it with the log.

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")